# Update

In [ ]:
#!/usr/bin/env python

try:
    # For Python 3.0 and later
    from urllib.request import urlopen
except ImportError:
    # Fall back to Python 2's urllib2
    from urllib2 import urlopen

import certifi
import json

def get_jsonparsed_data(url):
    """
    Receive the content of ``url``, parse it as JSON and return the object.

    Parameters
    ----------
    url : str

    Returns
    -------
    dict
    """
    response = urlopen(url, cafile=certifi.where())
    data = response.read().decode("utf-8")
    return json.loads(data)

In [ ]:
# url = ("https://financialmodelingprep.com/api/v3/historical-price-full/stock_dividend/AAPL?apikey=8941e8db3f939429e07397564861bc1a")
# new_data = get_jsonparsed_data(url)

In [ ]:
with open('/Users/Farid Srouji/Documents/Research/master/globaldata/companylist.txt') as f:
    companies = f.read().splitlines()

# del df
df = pd.DataFrame()
for company in companies:
    try:
        print("Company = "+company)
        url = ("https://financialmodelingprep.com/api/v3/historical-price-full/stock_dividend/%s?apikey=8941e8db3f939429e07397564861bc1a" % company)
        new_data = get_jsonparsed_data(url)
        # print(new_data)
        div_results = new_data["historical"]
        df_div= pd.DataFrame(data=div_results)
        df_div["company"]=company
        df_div.drop(columns={"label","dividend","recordDate"})
        df_div.loc[df_div["date"]>"2020-05-30"]
        frames = [df,df_div]
        df = pd.concat(frames)
    except:
        print("Company does not have div="+company)

In [ ]:
df = df.loc[df["date"]>"2020-05-31"]

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Dividends.pkl")

# View

In [ ]:
import pandas as pd
df_dividends=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Dividends.pkl', compression=None)

In [ ]:
df_dividends

In [ ]:
df_dividends = df_dividends[["date","dividend"]].groupby("date").count()#.plot(figsize=(20,5))
df_dividends["dividend_ln_return"] = np.log(df_dividends["dividend"]/df_dividends["dividend"].shift(periods=1)) 
df_dividends = df_dividends.reset_index().rename(columns={"date":"Date"})
df_dividends["Date"]= pd.to_datetime(df_dividends["Date"])

df_dividends = df_dividends.loc[df_dividends["Date"]>"2020-05-31"]#.sort(by="date")#.groupby("declarationDate").count()
df_dividends = df_dividends.loc[df_dividends["Date"]<"2021-12-21"]
df_dividends = df_dividends.sort_values(by="Date")
df_dividends.index = df_dividends["Date"]
df_dividends = df_dividends[["dividend_ln_return"]].dropna()

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

In [ ]:
df = df[["vix_ln_return"]]

In [ ]:
df = pd.merge(df, df_dividends, on='Date')#,how='outer'

In [ ]:
df = df[["vix_ln_return","dividend_ln_return"]]

In [ ]:
fig, ax1 = plt.subplots(figsize=(20,10))
l1 = ax1.plot(df['vix_ln_return'].ewm(10).mean(), color='#ff7f50')
ax1.set_xlabel("Date",fontsize=20)
ax1.set_title("VIX Ln Return and Dividend \n 2020-06-01 to "+PresentDate, size=26)
ax1.set_ylabel("VIX Ln Return", size=20)

ax2 = ax1.twinx()
l2 = ax2.plot(df['dividend_ln_return'].ewm(10).mean(), color='#0343DF')# color='#e50000')
# ax2.set_ylim([50, 100])
ax2.set_ylabel("Dividend Ln Return", size=20)

labels=['vix_ln_return', 'dividend_ln_return']
fig.legend([l1, l2], labels=labels, loc='right', bbox_to_anchor=(0.9, 0.85))

fig.autofmt_xdate()
fig = ax2.get_figure()

# fig.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images_press_release/timeserires_PressReleaseCount_vix_part3.png")

In [ ]:
import scipy.stats as stats
df = df[["vix_ln_return","dividend_ln_return"]]
# df = pd.read_csv('synchrony_sample.csv')
overall_pearson_r = df.corr().iloc[0,1]
print(f"Pandas computed Pearson r: {overall_pearson_r}")
# out: Pandas computed Pearson r: 0.2058774513561943

r, p = stats.pearsonr(df['vix_ln_return'].dropna(), df['dividend_ln_return'].dropna())
print(f"Scipy computed Pearson r: {r} and p-value: {p}")
# out: Scipy computed Pearson r: 0.20587745135619354 and p-value: 3.7902989479463397e-51

# Compute rolling window synchrony
f,ax=plt.subplots(figsize=(20,5))
df.rolling(window=30,center=True).median().plot(ax=ax)
ax.set(xlabel='Time',ylabel='Pearson r')
ax.set(title=f"Overall Pearson r = {np.round(overall_pearson_r,2)}");

In [ ]:
df.loc[df['dividend_ln_return']==np.inf]

mx = df.loc[df['dividend_ln_return'] != np.inf, 'dividend_ln_return'].max()
df['dividend_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['dividend_ln_return'] != -np.inf, 'dividend_ln_return'].min()
df['dividend_ln_return'].replace(-np.inf,mn,inplace=True)

In [ ]:
import scipy.stats as stats
df = df[["vix_ln_return","dividend_ln_return"]]
# df = pd.read_csv('synchrony_sample.csv')
overall_pearson_r = df.corr().iloc[0,1]
print(f"Pandas computed Pearson r: {overall_pearson_r}")
# out: Pandas computed Pearson r: 0.2058774513561943

r, p = stats.pearsonr(df['vix_ln_return'].dropna(), df['dividend_ln_return'].dropna())
print(f"Scipy computed Pearson r: {r} and p-value: {p}")
# out: Scipy computed Pearson r: 0.20587745135619354 and p-value: 3.7902989479463397e-51

# Compute rolling window synchrony
f,ax=plt.subplots(figsize=(20,5))
df.rolling(window=30,center=True).median().plot(ax=ax)
ax.set(xlabel='Time',ylabel='Pearson r')
ax.set(title=f"Overall Pearson r = {np.round(overall_pearson_r,2)}");

In [ ]:
import calplot
import numpy as np; np.random.seed(sum(map(ord, 'calplot')))
import pandas as pd
# df_dividends.index=df_dividends.date
df_dividends.index = pd.DatetimeIndex(df_dividends.index)
events = df_dividends.adjDividend#pd.Series(np.random.randn(len(days)), index=days)
fig, axes = calplot.calplot(events,cmap='YlOrRd', suptitle="Number of Companies Dividend Announcements")
df_dividends.sort_values(by="adjDividend").head(50)

In [ ]:
df_dividends1 = df_dividends1.groupby("company").count()

In [ ]:
df_dividends1

In [ ]:
df_dividends[["dividend","paymentDate"]].plot(figsize=(20,5))
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl', compression=None)
df_sentiment.loc["2020-05-31":][["PressReleaseCount"]].plot(figsize=(20,5))

In [ ]:
df_dividends1=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Dividends.pkl', compression=None)

In [ ]:
df_dividends1.groupby("company").count().mean()

In [ ]:
df_cal[["label"]].plot(figsize=(20,5))

In [ ]:
df_dividends.reset_index(drop=True, inplace=True)
df_cal = df_dividends.groupby("date").count()


In [ ]:
# df_cal.index=df_cal.date
df_cal.index = pd.DatetimeIndex(df_cal.index)
events = df_cal.adjDividend#pd.Series(np.random.randn(len(days)), index=days)
fig, axes = calplot.calplot(events,cmap='YlOrRd', suptitle="Dividends number of companies")

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

In [ ]:
new_df = pd.merge(df_dividends, df_articles,  how='left', left_on=['declarationDate','company'], right_on = ['Date','Company'])